# Predicting Attention Deficits in Children Using EEG and Behavioral Data
**Dataset:** OpenNeuro ds005507 — Healthy Brain Network (HBN-EEG) Release 3  
**Target:** CBCL Attention T-score → Binary (Low/High) + Continuous regression  

---
## Pipeline Overview
```
1. EEG Preprocessing  (filter → re-reference → ICA → epoch)
2. Feature Extraction  (band power, theta/alpha ratio, entropy, coherence, behavioral)
3. Model Training      (Classification + Regression, 5-fold CV)
4. Evaluation & Metrics
5. Visualization        (10 plots)
```

> **This notebook loads pre-computed results — no processing needed.** Run all cells to display metrics and plots.

---
## Cell 1 — Load All Saved Data

In [ ]:
import os, json, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap

from sklearn.metrics import (
    confusion_matrix, roc_curve, auc, ConfusionMatrixDisplay,
    mean_absolute_error, r2_score
)
from matplotlib.patches import Patch

plt.rcParams.update({'font.family': 'DejaVu Sans',
                     'axes.spines.top': False, 'axes.spines.right': False})
%matplotlib inline

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR       = os.getcwd()
DATA_PROC      = os.path.join(BASE_DIR, 'data', 'processed')
RESULTS_PLOTS  = os.path.join(BASE_DIR, 'results', 'plots')
RESULTS_MODELS = os.path.join(BASE_DIR, 'results', 'models')

ATTENTION_CUTOFF = 65.0
BANDS = {'delta': (1,4), 'theta': (4,8), 'alpha': (8,13), 'beta': (13,30), 'gamma': (30,45)}
META_COLS = ['subject_id','attention','attention_high',
             'p_factor','internalizing','externalizing','age','sex']

# ── Load everything ───────────────────────────────────────────────────────
feature_matrix = pd.read_csv(os.path.join(DATA_PROC, 'feature_matrix.csv'))
clf_results    = pd.read_csv(os.path.join(RESULTS_MODELS, 'classification_results.csv'))
reg_results    = pd.read_csv(os.path.join(RESULTS_MODELS, 'regression_results.csv'))
imp_df         = pd.read_csv(os.path.join(RESULTS_MODELS, 'feature_importance.csv'))
clf_preds      = pd.read_csv(os.path.join(RESULTS_MODELS, 'clf_predictions.csv'))
reg_preds      = pd.read_csv(os.path.join(RESULTS_MODELS, 'reg_predictions.csv'))

with open(os.path.join(RESULTS_MODELS, 'feature_names.json')) as f:
    feature_cols = json.load(f)

shap_data = np.load(os.path.join(RESULTS_MODELS, 'shap_data.npz'))
shap_vals = shap_data['shap_values']
X_shap    = shap_data['X_shap']

lc_data = np.load(os.path.join(RESULTS_MODELS, 'learning_curve.npz'))

best_clf = joblib.load(os.path.join(RESULTS_MODELS, 'best_classifier.pkl'))
best_reg = joblib.load(os.path.join(RESULTS_MODELS, 'best_regressor.pkl'))

feature_cols_matrix = [c for c in feature_matrix.columns if c not in META_COLS]
X       = feature_matrix[feature_cols_matrix].values
y_class = feature_matrix['attention_high'].values.astype(int)
y_reg   = feature_matrix['attention'].values.astype(float)

best_clf_name = clf_results.iloc[0]['Model']
best_reg_name = reg_results.iloc[0]['Model']

print('\u2713 All data loaded successfully')
print(f'  Feature matrix : {feature_matrix.shape[0]} subjects \u00d7 {len(feature_cols_matrix)} features')
print(f'  High attention : {y_class.sum()} ({y_class.mean()*100:.1f}%)')
print(f'  Low  attention : {(y_class==0).sum()} ({(1-y_class.mean())*100:.1f}%)')
print(f'  Best classifier: {best_clf_name}')
print(f'  Best regressor : {best_reg_name}')

---
## Cell 2 — Dataset Overview

In [ ]:
print('='*65)
print('  DATASET OVERVIEW')
print('='*65)
print(f'  Source        : OpenNeuro ds005507 (HBN-EEG Release 3)')
print(f'  Subjects      : {feature_matrix.shape[0]}')
print(f'  Age range     : {feature_matrix["age"].min():.1f} \u2013 {feature_matrix["age"].max():.1f} years')
print(f'  Features      : {len(feature_cols_matrix)}')
print(f'  \u2500 EEG spectral : band powers (5 bands \u00d7 19 channels = 95)')
print(f'  \u2500 Theta/Alpha  : ratio per channel + mean (20)')
print(f'  \u2500 Asymmetry    : frontal alpha asymmetry (3 pairs)')
print(f'  \u2500 Entropy      : spectral + sample entropy (38)')
print(f'  \u2500 Coherence    : alpha coherence (50 pairs)')
print(f'  \u2500 Behavioral   : RT mean/std/IQR + accuracy (8)')
print(f'\n  Label: CBCL Attention T-score')
print(f'  Binary cutoff : T \u2265 65 (clinical threshold)')
print(f'  Class balance : Low={int((y_class==0).sum())} / High={int(y_class.sum())}')
print('='*65)

feature_matrix[['subject_id','attention','attention_high','age','sex']].describe(include='all').round(2)

---
## Cell 3 — Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Exploratory Data Analysis', fontsize=16, fontweight='bold', y=1.01)

low_df  = feature_matrix[feature_matrix['attention_high']==0]
high_df = feature_matrix[feature_matrix['attention_high']==1]
COLORS  = {'Low': '#2196F3', 'High': '#F44336'}

# 1. Attention score distribution
ax = axes[0,0]
ax.hist(low_df['attention'],  bins=18, alpha=0.7, color=COLORS['Low'],  label='Low (T<65)')
ax.hist(high_df['attention'], bins=18, alpha=0.7, color=COLORS['High'], label='High (T\u226565)')
ax.axvline(65, color='black', linestyle='--', lw=1.5, label='Cutoff T=65')
ax.set_xlabel('CBCL Attention T-score'); ax.set_ylabel('Count')
ax.set_title('Attention Score Distribution', fontweight='bold'); ax.legend(fontsize=9)

# 2. Class balance pie
ax = axes[0,1]
counts = [len(low_df), len(high_df)]
ax.pie(counts, labels=[f'Low\n(n={counts[0]})', f'High\n(n={counts[1]})'],
       colors=[COLORS['Low'], COLORS['High']], autopct='%1.1f%%', startangle=90,
       textprops={'fontsize': 11})
ax.set_title('Class Balance', fontweight='bold')

# 3. Age distribution by class
ax = axes[0,2]
ax.hist(low_df['age'],  bins=15, alpha=0.7, color=COLORS['Low'],  label='Low')
ax.hist(high_df['age'], bins=15, alpha=0.7, color=COLORS['High'], label='High')
ax.set_xlabel('Age (years)'); ax.set_ylabel('Count')
ax.set_title('Age Distribution by Class', fontweight='bold'); ax.legend(fontsize=9)

# 4. Theta/alpha ratio by class
ax = axes[1,0]
if 'theta_alpha_ratio_mean' in feature_matrix.columns:
    ax.boxplot([low_df['theta_alpha_ratio_mean'].dropna(),
                high_df['theta_alpha_ratio_mean'].dropna()],
               tick_labels=['Low', 'High'], patch_artist=True,
               boxprops=dict(facecolor='#E3F2FD'),
               medianprops=dict(color='navy', linewidth=2))
    ax.set_ylabel('Theta/Alpha Ratio')
    ax.set_title('Theta/Alpha Ratio by Class\n(Key ADHD biomarker)', fontweight='bold')

# 5. p_factor vs attention
ax = axes[1,1]
scatter_df = feature_matrix.dropna(subset=['p_factor','attention'])
colors_map = scatter_df['attention_high'].map({0: COLORS['Low'], 1: COLORS['High']})
ax.scatter(scatter_df['p_factor'], scatter_df['attention'],
           c=colors_map, alpha=0.6, edgecolors='white', linewidths=0.3, s=50)
ax.axhline(65, color='gray', linestyle='--', lw=1, alpha=0.6)
ax.set_xlabel('p-factor (psychopathology)'); ax.set_ylabel('CBCL Attention T-score')
ax.set_title('p-factor vs Attention Score', fontweight='bold')
ax.legend(handles=[Patch(color=COLORS['Low'], label='Low'),
                   Patch(color=COLORS['High'], label='High')], fontsize=9)

# 6. Internalizing vs Externalizing
ax = axes[1,2]
scatter_df2 = feature_matrix.dropna(subset=['internalizing','externalizing'])
colors_map2 = scatter_df2['attention_high'].map({0: COLORS['Low'], 1: COLORS['High']})
ax.scatter(scatter_df2['internalizing'], scatter_df2['externalizing'],
           c=colors_map2, alpha=0.6, edgecolors='white', linewidths=0.3, s=50)
ax.set_xlabel('Internalizing Score'); ax.set_ylabel('Externalizing Score')
ax.set_title('Internalizing vs Externalizing', fontweight='bold')

plt.tight_layout()
plt.show()
print('\u2713 EDA complete')

---
## Cell 4 — Classification Results (5-Fold Stratified CV)

In [ ]:
print('='*70)
print('CLASSIFICATION: Low vs High Attention (5-Fold Stratified CV)')
print('='*70)
print(f'{"Model":<25} {"Accuracy":>10} {"F1":>10} {"AUC-ROC":>10} {"Kappa":>10}')
print('-'*70)
for _, r in clf_results.iterrows():
    print(f'{r["Model"]:<25} {r["Accuracy"]:>10.4f} {r["F1"]:>10.4f} '
          f'{r["AUC_ROC"]:>10.4f} {r["Cohen_Kappa"]:>10.4f}')
print('='*70)

best = clf_results.iloc[0]
print(f'\n\u2713 Best: {best["Model"]}  |  AUC={best["AUC_ROC"]:.4f} \u00b1 {best["AUC_ROC_std"]:.4f}')

clf_results.round(4)

---
## Cell 5 — Regression Results (5-Fold CV)

In [ ]:
print('='*65)
print('REGRESSION: Predict Continuous Attention Score (5-Fold CV)')
print('='*65)
print(f'{"Model":<25} {"MAE":>10} {"R2":>10} {"Pearson_r":>12}')
print('-'*65)
for _, r in reg_results.iterrows():
    print(f'{r["Model"]:<25} {r["MAE"]:>10.4f} {r["R2"]:>10.4f} {r["Pearson_r"]:>12.4f}')
print('='*65)

best_r = reg_results.iloc[0]
print(f'\n\u2713 Best: {best_r["Model"]}  |  R\u00b2={best_r["R2"]:.4f}  |  MAE={best_r["MAE"]:.4f}')

reg_results.round(4)

---
## Cell 6 — Feature Importance (Top 25)

In [ ]:
TOP = 25
top_imp = imp_df.head(TOP)

fig, ax = plt.subplots(figsize=(10, 7))
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, TOP))[::-1]
ax.barh(top_imp['feature'][::-1], top_imp['importance'][::-1],
        color=colors, edgecolor='white')
ax.set_xlabel('Feature Importance (Mean Decrease Gini)', fontsize=12)
ax.set_title(f'Top {TOP} Most Important Features\n(Random Forest)', fontsize=13, fontweight='bold')
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
plt.show()

print('\nTop 10 features:')
print(imp_df.head(10).to_string(index=False))

---
## Cell 7 — ROC Curves

In [ ]:
fpr, tpr, _ = roc_curve(clf_preds['true'], clf_preds['proba'])
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0,1], [0,1], 'k--', lw=1, label='Random (AUC=0.50)')
ax.plot(fpr, tpr, color='#1f77b4', lw=2.5,
        label=f'{best_clf_name} (AUC={roc_auc:.3f})')
ax.fill_between(fpr, tpr, alpha=0.1, color='#1f77b4')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve \u2014 Best Classifier (5-Fold CV)', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
plt.tight_layout()
plt.show()
print(f'AUC-ROC = {roc_auc:.4f}')

---
## Cell 8 — Confusion Matrix

In [ ]:
cm = confusion_matrix(clf_preds['true'], clf_preds['pred'])
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Low Attention', 'High Attention'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix \u2014 {best_clf_name}\n(5-Fold CV)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Model         : {best_clf_name}')
print(f'True Positives: {tp}  |  True Negatives: {tn}')
print(f'False Positives: {fp}  |  False Negatives: {fn}')
print(f'Sensitivity   : {sensitivity:.4f}  (recall for High Attention)')
print(f'Specificity   : {specificity:.4f}  (recall for Low Attention)')

---
## Cell 9 — Regression Scatter (Predicted vs Actual)

In [ ]:
true_r = reg_preds['true'].values
pred_r = reg_preds['pred'].values
r2  = r2_score(true_r, pred_r)
mae = mean_absolute_error(true_r, pred_r)
r   = np.corrcoef(true_r, pred_r)[0,1]

fig, ax = plt.subplots(figsize=(6, 5))
color_pts = np.where(true_r >= ATTENTION_CUTOFF, '#F44336', '#2196F3')
ax.scatter(true_r, pred_r, c=color_pts, alpha=0.65,
           edgecolors='white', linewidths=0.4, s=55)
mn = min(true_r.min(), pred_r.min()) - 2
mx = max(true_r.max(), pred_r.max()) + 2
ax.plot([mn, mx], [mn, mx], 'k--', lw=1.5)
ax.axvline(ATTENTION_CUTOFF, color='gray', ls=':', lw=1, alpha=0.5)
ax.axhline(ATTENTION_CUTOFF, color='gray', ls=':', lw=1, alpha=0.5)
ax.set_xlabel('Actual Attention T-score', fontsize=12)
ax.set_ylabel('Predicted Attention T-score', fontsize=12)
ax.set_title(f'Regression: Predicted vs Actual\n({best_reg_name})', fontsize=12, fontweight='bold')
ax.text(0.05, 0.93, f'R\u00b2 = {r2:.3f}\nMAE = {mae:.2f}\nr = {r:.3f}',
        transform=ax.transAxes, fontsize=11, va='top',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', alpha=0.9))
ax.legend(handles=[Patch(color='#F44336', label='High Attention'),
                   Patch(color='#2196F3', label='Low Attention'),
                   plt.Line2D([0],[0], color='k', ls='--', label='Perfect')],
          fontsize=9, loc='upper left')
plt.tight_layout()
plt.show()
print(f'R\u00b2 = {r2:.4f} | MAE = {mae:.4f} | Pearson r = {r:.4f}')

---
## Cell 10 — SHAP Feature Importance

In [ ]:
feat_names = feature_cols[:X_shap.shape[1]]

fig, ax = plt.subplots(figsize=(9, 7))
shap.summary_plot(shap_vals, X_shap, feature_names=feat_names,
                  show=False, max_display=20, plot_size=None)
plt.title('SHAP Feature Importance\n(High Attention class \u2014 positive SHAP = pushes toward High)',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print('\u2713 SHAP summary')

---
## Cell 11 — Learning Curve

In [ ]:
ts = lc_data['train_sizes']
tm, t_std = lc_data['train_mean'], lc_data['train_std']
vm, v_std = lc_data['val_mean'],   lc_data['val_std']

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(ts, tm, 'o-', color='#1976D2', label='Training AUC')
ax.fill_between(ts, tm-t_std, tm+t_std, alpha=0.15, color='#1976D2')
ax.plot(ts, vm, 's-', color='#F44336', label='Validation AUC')
ax.fill_between(ts, vm-v_std, vm+v_std, alpha=0.15, color='#F44336')
ax.set_xlabel('Training Set Size', fontsize=12)
ax.set_ylabel('AUC-ROC', fontsize=12)
ax.set_title('Learning Curve \u2014 Random Forest', fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.set_ylim([0.4, 1.05])
ax.axhline(0.5, color='gray', ls='--', lw=0.8, alpha=0.6)
plt.tight_layout()
plt.show()
print(f'Final validation AUC: {vm[-1]:.4f} \u00b1 {v_std[-1]:.4f}')

---
## Cell 12 — Model Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Model Comparison (5-Fold Cross Validation)', fontsize=14, fontweight='bold')

# Classification
ax = axes[0]
x  = np.arange(len(clf_results))
w  = 0.25
ax.bar(x-w, clf_results['Accuracy'], w, label='Accuracy', color='#42A5F5',
       yerr=clf_results['Accuracy_std'], capsize=4)
ax.bar(x,   clf_results['F1'],       w, label='F1 Score',  color='#66BB6A',
       yerr=clf_results['F1_std'], capsize=4)
ax.bar(x+w, clf_results['AUC_ROC'],  w, label='AUC-ROC',   color='#EF5350',
       yerr=clf_results['AUC_ROC_std'], capsize=4)
ax.set_xticks(x)
ax.set_xticklabels(clf_results['Model'], rotation=15, ha='right', fontsize=9)
ax.set_ylim([0, 1.15]); ax.set_ylabel('Score', fontsize=12)
ax.set_title('Classification Models', fontsize=12, fontweight='bold')
ax.axhline(0.5, color='gray', ls='--', lw=0.8, alpha=0.5)
ax.legend(fontsize=10)

# Regression
ax2 = axes[1]
x2  = np.arange(len(reg_results))
ax2.bar(x2-0.2, reg_results['R2'],        0.4, label='R\u00b2',        color='#AB47BC',
        yerr=reg_results['R2_std'], capsize=4)
ax2.bar(x2+0.2, reg_results['Pearson_r'], 0.4, label='Pearson r', color='#FF7043')
ax2.set_xticks(x2)
ax2.set_xticklabels(reg_results['Model'], rotation=15, ha='right', fontsize=9)
ax2.set_ylim([-0.15, 1.15]); ax2.set_ylabel('Score', fontsize=12)
ax2.set_title('Regression Models', fontsize=12, fontweight='bold')
ax2.axhline(0, color='gray', ls='--', lw=0.8, alpha=0.5)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()
print('\u2713 Model comparison')

---
## Cell 13 — Feature Correlation Heatmap

In [ ]:
top15 = imp_df['feature'].head(15).tolist()
available = [f for f in top15 if f in feature_matrix.columns]
heat_df   = feature_matrix[available + ['attention']].dropna()
corr_mat  = heat_df.corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_mat, dtype=bool))
sns.heatmap(corr_mat, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5,
            ax=ax, annot_kws={'size': 8})
ax.set_title('Feature Correlation Heatmap\n(Top 15 Features + Attention Score)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('\u2713 Correlation heatmap')

---
## Cell 14 — EEG Band Power Comparison

In [ ]:
band_means = {}
for band in BANDS:
    band_cols = [c for c in feature_matrix.columns if c.startswith(f'{band}_ch')]
    if band_cols:
        feature_matrix[f'{band}_mean'] = feature_matrix[band_cols].mean(axis=1)
        band_means[band] = feature_matrix.groupby('attention_high')[f'{band}_mean'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('EEG Band Power: High vs Low Attention', fontsize=14, fontweight='bold')

# Bar chart
ax = axes[0]
bands_list = list(band_means.keys())
low_vals  = [band_means[b][0] for b in bands_list]
high_vals = [band_means[b][1] for b in bands_list]
x = np.arange(len(bands_list))
ax.bar(x-0.2, low_vals,  0.4, label='Low Attention',  color='#2196F3', alpha=0.85)
ax.bar(x+0.2, high_vals, 0.4, label='High Attention', color='#F44336', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([b.capitalize() for b in bands_list], fontsize=12)
ax.set_ylabel('Mean Band Power (\u00b5V\u00b2/Hz)', fontsize=11)
ax.set_title('Mean Band Power by Group', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

# Theta/alpha ratio violin
ax2 = axes[1]
if 'theta_alpha_ratio_mean' in feature_matrix.columns:
    data_low  = feature_matrix[feature_matrix['attention_high']==0]['theta_alpha_ratio_mean'].dropna()
    data_high = feature_matrix[feature_matrix['attention_high']==1]['theta_alpha_ratio_mean'].dropna()
    vp = ax2.violinplot([data_low, data_high], positions=[1,2], showmedians=True)
    for pc, color in zip(vp['bodies'], ['#2196F3','#F44336']):
        pc.set_facecolor(color); pc.set_alpha(0.7)
    ax2.set_xticks([1,2])
    ax2.set_xticklabels(['Low Attention','High Attention'], fontsize=11)
    ax2.set_ylabel('Theta/Alpha Ratio', fontsize=11)
    ax2.set_title('Theta/Alpha Ratio Distribution\n(Key ADHD biomarker)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()
print('\u2713 Band power comparison')

---
## Cell 15 — Final Summary

In [ ]:
cm = confusion_matrix(clf_preds['true'], clf_preds['pred'])
tn, fp, fn, tp = cm.ravel()

print('='*70)
print('  FINAL RESULTS SUMMARY')
print('  Predicting Attention Deficits from EEG and Behavioral Data')
print('='*70)

print(f'\n  Dataset       : OpenNeuro ds005507 (HBN-EEG Release 3)')
print(f'  Subjects      : {feature_matrix.shape[0]}')
print(f'  Features      : {len(feature_cols_matrix)} (EEG spectral + connectivity + behavioral)')
print(f'  High attention: {int(y_class.sum())} ({y_class.mean()*100:.1f}%)')
print(f'  Low  attention: {int((y_class==0).sum())} ({(1-y_class.mean())*100:.1f}%)')

best_c = clf_results.iloc[0]
print(f'\n  CLASSIFICATION (Low vs High Attention, T\u226565 cutoff)')
print(f'  Best Model    : {best_c["Model"]}')
print(f'  Accuracy      : {best_c["Accuracy"]:.4f} \u00b1 {best_c["Accuracy_std"]:.4f}')
print(f'  F1 Score      : {best_c["F1"]:.4f} \u00b1 {best_c["F1_std"]:.4f}')
print(f'  AUC-ROC       : {best_c["AUC_ROC"]:.4f} \u00b1 {best_c["AUC_ROC_std"]:.4f}')
print(f'  Cohen Kappa   : {best_c["Cohen_Kappa"]:.4f}')
print(f'  Sensitivity   : {tp/(tp+fn):.4f}')
print(f'  Specificity   : {tn/(tn+fp):.4f}')

best_r = reg_results.iloc[0]
print(f'\n  REGRESSION (Predict continuous CBCL attention score)')
print(f'  Best Model    : {best_r["Model"]}')
print(f'  MAE           : {best_r["MAE"]:.4f} \u00b1 {best_r["MAE_std"]:.4f}')
print(f'  R\u00b2            : {best_r["R2"]:.4f}')
print(f'  Pearson r     : {best_r["Pearson_r"]:.4f}')

print(f'\n  Top 5 Predictive Features:')
for _, row in imp_df.head(5).iterrows():
    print(f'    {row["feature"]:<35} {row["importance"]:.4f}')

print('\n' + '='*70)